In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import pandas as pd
import cv2
import string
import shutil
import numpy as np
import pyheif
from PIL import Image
from PIL import Image
import pillow_heif
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

We want to predict given an image of a handwritten letter which letter it is. We specify that the letter be lowercase, written in cursive, and be black against a white background. To do this, we will train a neural network model that reads the image and creates a prediction.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

We earlier created the data ourselves by having everyone in the class handwrite each letter of the alphabet. The images for these were then compiled and put into a folder for us. We may also gather additional data by looking at the NIST dataset.

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

      Sex  Class   Overall     Adult     Child
0    male      1  0.368852  0.371134  0.360000
1    male      2  0.157407  0.068182  0.550000
2    male      3  0.135447  0.133333  0.138686
3  female      1  0.968085  0.974026  0.941176
4  female      2  0.921053  0.903226  1.000000
5  female      3  0.500000  0.417910  0.571429


In [17]:
base_folder = 'Cursive_Explore'
letters = list(string.ascii_lowercase)

'''
for folder in sorted(os.listdir(base_folder)):
    folder_path = os.path.join(base_folder, folder)

    if not os.path.isdir(folder_path):
        continue

    files = sorted(os.listdir(folder_path))

    for i, filename in enumerate(files):
        if i >= len(letters):
            break

        old_path = os.path.join(folder_path, filename)
        if os.path.isfile(old_path):
            ext = os.path.splitext(filename)[1]
            new_name = f"{letters[i]}{ext}"
            new_path = os.path.join(folder_path, new_name)
            os.rename(old_path, new_path)

    # Rename files in each folder
'''

'''
for item in os.listdir(base_folder):
    subfolder_path = os.path.join(base_folder, item)
    if os.path.isdir(subfolder_path):
        for filename in os.listdir(subfolder_path):
            file_path = os.path.join(subfolder_path, filename)
            if os.path.isfile(file_path):
                dest_path = os.path.join(base_folder, filename)
                if os.path.exists(dest_path):
                    name, ext = os.path.splitext(filename)
                    counter = 1
                    while True:
                        new_name = f"{name}_{counter}{ext}"
                        new_dest = os.path.join(base_folder, new_name)
                        if not os.path.exists(new_dest):
                            dest_path = new_dest
                            break
                        counter += 1
                os.rename(file_path, dest_path)
        os.rmdir(subfolder_path)
    # Get files out of their subfolders
'''

for filename in os.listdir(base_folder):
    file_path = os.path.join(base_folder, filename)
    if os.path.isfile(file_path):
        folder_name = filename[0].lower() + "_letters"
        folder_path = os.path.join(base_folder, folder_name)
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
        new_path = os.path.join(folder_path, filename)
        os.rename(file_path, new_path)
    # Create separate folder for each individual letter

In general, we aren't looking for that many patterns here, since we are already sure we are using a neural network and our strategy will be to get the neural network to pick up on these patterns. For this step, we created folders that contained every instance of each individual letter. We then looked through the folders to search for patterns manually.

At this point, we found a couple of letters that were labeled incorrectly, which we may be forced to fix. Looking through the letters, it seems like most of the letters are relatively well behaved, except for letters like f, k, z. We also saw that many of the images contained backgrounds that needed to be cut out, which is something we will have to do during data preparation.

# 4.Prepare the Data


Apply any data transformations and explain what and why


In [30]:
base_folder = 'Cursive_Train'
'''
for item in os.listdir(base_folder):
    subfolder_path = os.path.join(base_folder, item)
    if os.path.isdir(subfolder_path):
        for filename in os.listdir(subfolder_path):
            file_path = os.path.join(subfolder_path, filename)
            if os.path.isfile(file_path):
                dest_path = os.path.join(base_folder, filename)
                if os.path.exists(dest_path):
                    name, ext = os.path.splitext(filename)
                    counter = 1
                    while True:
                        new_name = f"{name}_{counter}{ext}"
                        new_dest = os.path.join(base_folder, new_name)
                        if not os.path.exists(new_dest):
                            dest_path = new_dest
                            break
                        counter += 1
                os.rename(file_path, dest_path)
        os.rmdir(subfolder_path)
    # Get files out of their subfolders
'''

for file in os.listdir('Cursive_Train'):
    file_path = os.path.join(base_folder, file)
    if os.path.isfile(file_path):
        img = cv2.imread(file_path)
        if img is None:
            continue
        resized = cv2.resize(img, (128, 256))
        name, _ = os.path.splitext(file)
        output_path = os.path.join("images", f"standardized__{name}.jpg")
        cv2.imwrite(output_path, resized)


One of the issues we already had when looking through the data was inconsistent image formats. This code reads every image using opencv and writes it in a standard format to a different folder. We also standardize the image size. This is extremely important because it fixes the number of inputs to our neural network, which is clearly necessary for it to be able to train properly.

In [8]:
def rename(base_path):
    for root, dirs, files in os.walk(base_path):
        visible_files = [f for f in files if not f.startswith('.')]
        if not visible_files:
            continue
        if len(visible_files) == 26:
            visible_files.sort()
            for letter, old_name in zip(string.ascii_lowercase, visible_files):
                old_path = os.path.join(root, old_name)
                ext = os.path.splitext(old_name)[1]
                new_name = f"{letter}{ext}"
                new_path = os.path.join(root, new_name)
                os.rename(old_path, new_path)
        else:
            for old_name in visible_files:
                old_path = os.path.join(root, old_name)
                ext = os.path.splitext(old_name)[1]
                new_name = f"IGNORE{ext}"
                new_path = os.path.join(root, new_name)
                os.rename(old_path, new_path)

# rename('Train')

def move(base_path):
    images_dir = 'images'
    counters = {letter: 1 for letter in string.ascii_lowercase}
    for root, _, files in os.walk(base_path):
        for fname in files:
            if fname.startswith('.') or fname.startswith('IGNORE'):
                continue
            letter = fname[0].lower()
            ext = os.path.splitext(fname)[1]
            old_path = os.path.join(root, fname)
            if letter not in counters:
                continue
            count = counters[letter]
            new_name = f"{letter}{count}{ext}"
            new_path = os.path.join(images_dir, new_name)
            shutil.move(old_path, new_path)
            counters[letter] += 1

# move('Train')

def standardize(directory):
    for fname in os.listdir(directory):
        if fname.startswith('.'):
            continue
        fpath = os.path.join(directory, fname)
        if os.path.isdir(fpath):
            continue
        root, ext = os.path.splitext(fname)
        ext = ext.lower().strip()
        if not ext:
            new_name = f"{fname}.unknown" if not fname.endswith(".unknown") else fname
            new_path = os.path.join(directory, new_name)
            if new_path != fpath:
                os.rename(fpath, new_path)
                print(f"🔤 Renamed {fname} → {new_name}")
            fpath = new_path
            fname = new_name
            root, ext = os.path.splitext(fname)
            ext = ext.lower().strip()
        out_path = os.path.join(directory, f"{root}.png")
        if os.path.exists(out_path):
            continue
        try:
            if ext in ['.jpg', '.jpeg', '.png']:
                img = cv2.imread(fpath)
                if img is None:
                    raise ValueError("cv2 failed to read image")
            elif ext in ['.heic', '.heif', '.unknown']:
                heif_file = pyheif.read(fpath)
                image = Image.frombytes(
                    heif_file.mode,
                    heif_file.size,
                    heif_file.data,
                    "raw",
                    heif_file.mode
                )
                img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            else:
                continue
            cv2.imwrite(out_path, img)
            if os.path.exists(fpath):
                os.remove(fpath)
        except Exception as e:
            try:
                os.remove(fpath)
                print('fail')
            except:
                print('fail')

# standardize('images')

def keep_pngs(directory):
    for fname in os.listdir(directory):
        fpath = os.path.join(directory, fname)
        if os.path.isdir(fpath):
            continue
        if not fname.lower().endswith('.png'):
            try:
                os.remove(fpath)
            except Exception as e:
                print('fail')

# keep_pngs('images')

def binarize_images(input_dir, output_dir='binarized', threshold=100, min_size=50):
    os.makedirs(output_dir, exist_ok=True)
    for fname in os.listdir(input_dir):
        if not fname.lower().endswith('.png'):
            continue
        fpath = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)
        img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print('fail')
            continue
        _, bin_img = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY_INV)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_img, connectivity=8)
        bin_clean = np.zeros_like(bin_img)
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= min_size:
                bin_clean[labels == i] = 255
        cv2.imwrite(out_path, bin_clean)

# binarize_images('images')

def resize(input_dir='binarized', output_dir='resized', size=512):
    os.makedirs(output_dir, exist_ok=True)
    for fname in os.listdir(input_dir):
        if not fname.lower().endswith('.png'):
            continue
        fpath = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)
        img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        h, w = img.shape
        scale = size / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((size, size), dtype=np.uint8)
        x_offset = (size - new_w) // 2
        y_offset = (size - new_h) // 2
        canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
        cv2.imwrite(out_path, canvas)

# resize()

def crop(input_dir='resized', output_dir='cropped', min_component_size=30, pad_to_square=True):
    os.makedirs(output_dir, exist_ok=True)
    for fname in os.listdir(input_dir):
        if not fname.lower().endswith('.png'):
            continue
        fpath = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)
        img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(img, connectivity=8)
        mask = np.zeros_like(img)
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= min_component_size:
                mask[labels == i] = 255
        coords = cv2.findNonZero(mask)
        if coords is None:
            continue
        x, y, w, h = cv2.boundingRect(coords)
        cropped = img[y:y+h, x:x+w]
        if pad_to_square:
            size = max(w, h)
            canvas = np.zeros((size, size), dtype=np.uint8)
            x_offset = (size - w) // 2
            y_offset = (size - h) // 2
            canvas[y_offset:y_offset+h, x_offset:x_offset+w] = cropped
            cropped = canvas
        cv2.imwrite(out_path, cropped)

# crop()

# resize(input_dir='cropped', output_dir='test', size=32)

def subfolders(input_dir='test'):
    for fname in os.listdir(input_dir):
        if not fname.lower().endswith('.png'):
            continue
        first_letter = fname[0].lower()
        folder_path = os.path.join(input_dir, first_letter)
        os.makedirs(folder_path, exist_ok=True)
        src_path = os.path.join(input_dir, fname)
        dst_path = os.path.join(folder_path, fname)
        shutil.move(src_path, dst_path)

# subfolders()

The above code performs much stronger preprocessing. We convert all images to pngs as before, and now binarize each image, crop to just the letter, and then resize to 32x32 images. The test data is written to the directory 'test'.

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [5]:
# Define the train/test data

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

data_dir = 'test'
batch_size = 16
num_epochs = 20
learning_rate = 0.001
num_classes = 26

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

device = torch.device('cpu')

print(dataset.class_to_idx)

{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25}


In [7]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(32*8*8, 64)
        self.fc2   = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    test_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Test: {test_acc:.4f}")

torch.save(model.state_dict(), 'model.pth')

Epoch [1/20], Loss: 3.2733, Test: 0.0405
Epoch [2/20], Loss: 3.2157, Test: 0.0405
Epoch [3/20], Loss: 3.0373, Test: 0.0405
Epoch [4/20], Loss: 2.8087, Test: 0.0811
Epoch [5/20], Loss: 2.5788, Test: 0.1351
Epoch [6/20], Loss: 2.3527, Test: 0.1216
Epoch [7/20], Loss: 2.1310, Test: 0.1757
Epoch [8/20], Loss: 1.9401, Test: 0.2027
Epoch [9/20], Loss: 1.7204, Test: 0.1622
Epoch [10/20], Loss: 1.5292, Test: 0.1892
Epoch [11/20], Loss: 1.3377, Test: 0.1892
Epoch [12/20], Loss: 1.2456, Test: 0.1486
Epoch [13/20], Loss: 1.1209, Test: 0.2162
Epoch [14/20], Loss: 0.9651, Test: 0.2027
Epoch [15/20], Loss: 0.8605, Test: 0.2297
Epoch [16/20], Loss: 0.7910, Test: 0.2297
Epoch [17/20], Loss: 0.8052, Test: 0.2162
Epoch [18/20], Loss: 0.7167, Test: 0.2162
Epoch [19/20], Loss: 0.6889, Test: 0.2297
Epoch [20/20], Loss: 0.6224, Test: 0.2297


We created a neural net and trained it on the data. We ended with around 23% accuracy.

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


In [11]:
model.load_state_dict(torch.load('model.pth'))
model.to(device)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
fine_tune_epochs = 20

for epoch in range(fine_tune_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    test_acc = correct / total

    print(f"[Fine-tune Epoch {epoch+1}/{fine_tune_epochs}] Loss: {epoch_loss:.4f}, Test: {test_acc:.4f}")

torch.save(model.state_dict(), 'model_finetuned.pth')

[Fine-tune Epoch 1/20] Loss: 0.5673, Test: 0.2162
[Fine-tune Epoch 2/20] Loss: 0.5594, Test: 0.2297
[Fine-tune Epoch 3/20] Loss: 0.5534, Test: 0.2297
[Fine-tune Epoch 4/20] Loss: 0.5486, Test: 0.2432
[Fine-tune Epoch 5/20] Loss: 0.5443, Test: 0.2432
[Fine-tune Epoch 6/20] Loss: 0.5404, Test: 0.2432
[Fine-tune Epoch 7/20] Loss: 0.5370, Test: 0.2432
[Fine-tune Epoch 8/20] Loss: 0.5341, Test: 0.2432
[Fine-tune Epoch 9/20] Loss: 0.5312, Test: 0.2432
[Fine-tune Epoch 10/20] Loss: 0.5286, Test: 0.2432
[Fine-tune Epoch 11/20] Loss: 0.5262, Test: 0.2297
[Fine-tune Epoch 12/20] Loss: 0.5241, Test: 0.2297
[Fine-tune Epoch 13/20] Loss: 0.5218, Test: 0.2297
[Fine-tune Epoch 14/20] Loss: 0.5198, Test: 0.2297
[Fine-tune Epoch 15/20] Loss: 0.5180, Test: 0.2297
[Fine-tune Epoch 16/20] Loss: 0.5162, Test: 0.2297
[Fine-tune Epoch 17/20] Loss: 0.5143, Test: 0.2297
[Fine-tune Epoch 18/20] Loss: 0.5127, Test: 0.2297
[Fine-tune Epoch 19/20] Loss: 0.5110, Test: 0.2297
[Fine-tune Epoch 20/20] Loss: 0.5093, Te

We fine tune the model by decreasing the learning rate from 10^-3 to 10^-5, and training from the existing weights. This slightly increases the accuracy to around 24.32%.

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


In order to solve the problem of identifying a handwritten cursive letter, we created a neural network that trained on processed data. We processed the data by standardizing image format, binarizing to grab just the strokes in the letters, cropping to remove unwanted marks, and then resizing to a standard format. We then trained a convolutional neural network on this data. After fine tuning, we ended with around a 24% accuracy. While this is far from 100% accuracy, it is significantly larger than guessing at random. We were able to make partial progress towards accurately identifying letters.

# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [2]:
def inference(file_path, final_size=32, threshold=100, min_size=50, min_component_size=30):
    import os
    import pandas as pd
    import cv2
    import string
    import shutil
    import numpy as np
    import pyheif
    from PIL import Image
    from PIL import Image
    import pillow_heif
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import transforms, datasets
    from torch.utils.data import DataLoader, random_split

    class CNN(nn.Module):
        def __init__(self, num_classes=26):
            super(CNN, self).__init__()
            self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
            self.pool1 = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
            self.pool2 = nn.MaxPool2d(2, 2)
            self.fc1   = nn.Linear(32*8*8, 64)
            self.fc2   = nn.Linear(64, num_classes)
            self.relu = nn.ReLU()

        def forward(self, x):
            x = self.relu(self.conv1(x))
            x = self.pool1(x)
            x = self.relu(self.conv2(x))
            x = self.pool2(x)
            x = x.view(x.size(0), -1)
            x = self.relu(self.fc1(x))
            x = self.fc2(x)
            return x
    
    root, ext = os.path.splitext(file_path)
    ext = ext.lower().strip()
    out_path = f"{root}.png"
    if not file_path.lower().endswith('.png'):
        try:
            if ext in ['.jpg', '.jpeg', '.png']:
                img = cv2.imread(file_path)
            elif ext in ['.heic', '.heif', '.unknown']:
                heif_file = pyheif.read(file_path)
                image = Image.frombytes(
                    heif_file.mode,
                    heif_file.size,
                    heif_file.data,
                    "raw",
                    heif_file.mode
                )
                img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            else:
                raise ValueError("File Type")
            cv2.imwrite(out_path, img)
            file_path = out_path
        except Exception as e:
            raise RuntimeError(f"Conversion Failure")
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise RuntimeError("Failed to Read")
    _, bin_img = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY_INV)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_img, connectivity=8)
    bin_clean = np.zeros_like(bin_img)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            bin_clean[labels == i] = 255
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_clean, connectivity=8)
    mask = np.zeros_like(bin_clean)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_component_size:
            mask[labels == i] = 255
    coords = cv2.findNonZero(mask)
    if coords is None:
        raise RuntimeError("No Components")
    x, y, w, h = cv2.boundingRect(coords)
    cropped = bin_clean[y:y+h, x:x+w]
    size = max(w, h)
    canvas = np.zeros((size, size), dtype=np.uint8)
    x_offset = (size - w) // 2
    y_offset = (size - h) // 2
    canvas[y_offset:y_offset+h, x_offset:x_offset+w] = cropped
    final_img = cv2.resize(canvas, (final_size, final_size), interpolation=cv2.INTER_AREA)
    img_tensor = torch.tensor(final_img, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
    
    device = torch.device('cpu')
    num_classes = 26
    model = CNN(num_classes=num_classes).to(device)
    model.load_state_dict(torch.load('model_finetuned.pth', map_location=device))
    model.eval()

    img_tensor = img_tensor.to(device)
    with torch.no_grad():
        output = model(img_tensor)
        predicted_idx = torch.argmax(output, dim=1).item()
        predicted_letter = chr(predicted_idx + ord('a'))
    return predicted_letter

In [3]:
print(inference('Cursive_Explore/a_letters/a.jpeg'))

u
